# Bonus 03 — Text ReAct from Scratch
**Optional | After Lab 1B and Lab 2 | Colab CPU | OpenAI API key**

Lab 1B already *is* ReAct, with JSON `tool_calls`. This bonus shows the **older text format** so papers and legacy agents make sense:

```
Thought: I need memory for a 7B INT4 model.
Action: estimate_memory: 7, int4
PAUSE
Observation: {"gb": 3.5}
Answer: ...
```

Regex parsing is fragile. That is the lesson. Production stays on structured tools.

> No `eval()`. The calculator only adds numbers. Tools are deployment-themed, not dog weights.


In [ ]:
import sys
%pip install -q uv
!uv pip install -q --python {sys.executable} openai python-dotenv

In [ ]:
import os
try:
    from google.colab import userdata          # Colab: read the Secret you added
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv             # local: read .env in the repo root
    load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "Add OPENAI_API_KEY as a Colab Secret or to .env"

OPENAI_API_KEY  = os.environ["OPENAI_API_KEY"]
OPENAI_BASE_URL = "https://api.openai.com/v1"
DEFAULT_MODEL   = "gpt-4o-mini"

from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL)
print(f"Ready — {DEFAULT_MODEL} at {OPENAI_BASE_URL}")

## 1. Three tiny tools (Python you trust)


In [ ]:
import json
import re

def estimate_memory(params_b: float, precision: str) -> str:
    table = {"fp32": 4.0, "fp16": 2.0, "int8": 1.0, "int4": 0.5, "nf4": 0.5}
    key = precision.lower()
    if key not in table:
        return json.dumps({"error": f"unsupported precision {precision}"})
    return json.dumps({"params_b": params_b, "precision": key, "gb": round(params_b * table[key], 2)})

def kb_lookup(topic: str) -> str:
    kb = {"qlora": "QLoRA = NF4 quantized base + LoRA adapters. Train ~1% of parameters.",
          "vllm":  "vLLM uses PagedAttention and continuous batching for high throughput serving.",
          "rag":   "RAG retrieves chunks at inference and grounds the prompt. Not a fine-tune."}
    return kb.get(topic.lower().strip(), f"No note stored for {topic!r}. Try qlora, vllm, or rag.")

def add_numbers(a: float, b: float) -> str:
    return json.dumps({"sum": a + b})

print(estimate_memory(7, "int4"))
print(kb_lookup("qlora"))

In text ReAct the model writes arguments as **one string** after the tool name, so we need a parser. `_two` splits on a comma and tries to make numbers. This parser is the fragile part; JSON `tool_calls` (Lab 1B) removed it.

In [ ]:
def _two(raw: str):
    parts = [p.strip() for p in raw.split(",")]
    if len(parts) != 2:
        raise ValueError("expected two comma-separated arguments")
    try:
        return float(parts[0]), float(parts[1])
    except ValueError:
        return parts[0], parts[1]

TOOLS = {
    "estimate_memory": lambda raw: estimate_memory(*_two(raw)),
    "kb_lookup":       lambda raw: kb_lookup(raw.strip()),
    "add_numbers":     lambda raw: add_numbers(*_two(raw)),
}

print(TOOLS["estimate_memory"]("7, int4"))
print(TOOLS["add_numbers"]("2, 3.5"))

## 2. The text contract

The system prompt is the entire protocol. If the model adds a period in the wrong place, regex misses the Action. Watch for that.


In [ ]:
SYSTEM = '''You solve questions using this format and nothing else until you can Answer:
Thought: <one sentence>
Action: <tool>: <args>
PAUSE

Tools:
- estimate_memory: <params_b>, <precision>
- kb_lookup: <qlora|vllm|rag>
- add_numbers: <a>, <b>

When you have an Observation and can finish:
Thought: <one sentence>
Answer: <final>'''

print("system prompt ready")


## 3. The loop

Max 5 turns. Parse one `Action:` line, run the tool, send `Observation:` back as the next user message.


In [ ]:
ACTION_RE = re.compile(r"^Action:\s*(\w+):\s*(.*)$", re.MULTILINE)

def text_react(question: str, max_turns: int = 5) -> str:
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question},
    ]
    for turn in range(1, max_turns + 1):
        text = client.chat.completions.create(
            model=DEFAULT_MODEL, temperature=0, messages=messages
        ).choices[0].message.content
        print(f"--- turn {turn} ---")
        print(text)
        messages.append({"role": "assistant", "content": text})
        if re.search(r"^Answer:", text, re.MULTILINE) and not ACTION_RE.search(text):
            return text
        match = ACTION_RE.search(text)
        if not match:
            print("No Action line parsed — this is why JSON tool_calls won.")
            return text
        name, raw = match.group(1), match.group(2)
        fn = TOOLS.get(name)
        obs = fn(raw) if fn else f"Unknown tool {name}"
        print("Observation:", obs)
        messages.append({"role": "user", "content": f"Observation: {obs}"})
    return "Stopped at max_turns"

print(text_react("How much memory does a 7B INT4 model need, and what is QLoRA in one sentence?"))


**Checkpoint:** you should see `estimate_memory` then `kb_lookup` (or one combined thought), then an Answer. If parsing fails, that is the teaching moment.

| | This notebook | Lab 1B |
|---|---|---|
| Action | `Action: name: args` | JSON `tool_calls` |
| Parse | Regex | API-typed arguments |
| Ship it? | No | Yes |

## MCP, in one screen

[Model Context Protocol](https://modelcontextprotocol.io) is a **standard** for exposing tools (and resources) so every agent host does not rewrite `get_flight`. You still write the Python. MCP is the USB port; function calling is the electricity.

You do **not** need to build an MCP server in this course. Know the name for when a vendor says “we speak MCP.”

## Bonus 03 complete

Next: [Lab 6 RAG](../06_RAG_Pipeline/README.md) if you are still on Day 2, or [Bonus 01](01_function_calling.ipynb) for DuckDB multi-tool calling.
